In [ ]:
# 1. 环境依赖安装，首次安装后无需重复安装
!pip install -r requirements.txt

In [ ]:
# 2. CPU、NPU / CPU、NPU、NUMA拓扑关系可视化
!python3 topology_visualizer.py

from IPython.display import IFrame
IFrame("ascend_topo.html", width="100%", height="850px")

In [ ]:
# 3. 当前CPU间模型主要运行线程可视化
from key_pstree_visualizer import KeyPstreeVisualizer

keyPstreeVisualizer = KeyPstreeVisualizer()
# 用户可以输入 PID 或进程名/正则，例如：roots = keyPstreeVisualizer.build_pstree(extra_input=["python", 1234])
roots = keyPstreeVisualizer.build_pstree()

print("=== 全部进程树 ===")
keyPstreeVisualizer.print_tree(roots)

# 交互式搜索
keyPstreeVisualizer.interactive_search(roots)

In [ ]:
# 4. 模型主要运行线程绑核建议
from cpu_binding_suggestion import CpuBindingSuggestion
from IPython.display import Markdown, display

display(Markdown(CpuBindingSuggestion.generate_markdown()))

In [ ]:
# 5. 绑核后模型主要运行线程可视化校验
'''
usage: cpu_affinity_data_collection.py [-h] [--csv]
                                       [--npu-process [NPU_PROCESS ...]]
                                       [--datawork-process [DATAWORK_PROCESS ...]]

CPU Binding Validation

options:
  -h, --help            show this help message and exit
  --csv                 输出 CSV 格式
  --npu-process [NPU_PROCESS ...]
                        npu进程额外关注的线程名
  --datawork-process [DATAWORK_PROCESS ...]
                        datawork要扫描的进程/线程关键词 (不输入则不扫描)
'''
# 导出为csv格式：!python3 cpu_affinity_data_collection.py --csv > test.csv
# 直接打屏输出：!python3 cpu_affinity_data_collection.py --npu-process hccp_connect --datawork-process hccp_epoll

import os
from cpu_affinity_data_visualizer import run_notebook_app

# ==========================================================
# 快捷配置区：只需修改这里
# ==========================================================
# 1. NPU 相关线程关键字 (例如: 'hccp'，不填则只扫描默认内容)
NPU_PROCESS = "CommWorker DataWorker" 

# 2. 业务进程/线程关键字 (不填则不扫描)
DATAWORK_PROCESS = "" 

# 3. 数据保存文件名
OUTPUT_FILE = "affinity_data.csv"
# ==========================================================
npu_part = f"--npu-process {NPU_PROCESS}" if NPU_PROCESS else ""
dw_part = f"--datawork-process {DATAWORK_PROCESS}" if DATAWORK_PROCESS else ""
collect_cmd = f"python3 cpu_affinity_data_collection.py --csv {npu_part} {dw_part} > {OUTPUT_FILE}"

print(f"开始执行采集...")
print(f"指令: {collect_cmd}")
os.system(collect_cmd)

if os.path.exists(OUTPUT_FILE) and os.path.getsize(OUTPUT_FILE) > 0:
    print(f"采集成功！正在加载可视化界面...")
    run_notebook_app(OUTPUT_FILE)
else:
    print(f"错误：未能生成数据")